# California Wildfires and Pollution (PM2.5) on California Prison and Detention Facilities

This notebook contains the logic for finalizing the facilities data, from restoring information from raw files -> geocoding addresses -> consolidating information together to finally create polygons in QGIS and finalize data in a jupyter notebook.

All data from this notebook was gathered from: 

CDCR Facilities and Conservation Camps Data: 
https://www.cdcr.ca.gov/adult-operations/list-of-adult-institutions/

CDCR Monthly Adult Facility Population Data: 
https://www.cdcr.ca.gov/research/https-www-cdcr-ca-gov-research-monthly-total-population-report-archive-2019-to-2022/

California Ice Detention Facilities and Population Data:
https://tracreports.org/immigration/detentionstats/facilities.html

## Data Cleaning and Creating a Consolidated DataFrame for all Facilities and Detention Centers

In [1]:
import os
import re
import datetime 
import pdfplumber
from pathlib import Path

import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

import geopandas as gpd

from datetime import datetime
from dotenv import load_dotenv, find_dotenv
from sklearn import linear_model as lm

from geopy.geocoders import Nominatim
from geopy.geocoders import GoogleV3
from geopy.extra.rate_limiter import RateLimiter

In [2]:
# Loading environment variables in .env

load_dotenv()

GOOGLE_GEOCODE_API_KEY = os.getenv("GOOGLE_GEOCODE_API_KEY")

In [70]:
# Obtaining Path files to the data in data folder

directory_name = os.getcwd()
directory_path = Path(directory_name)

# CDCR txt file path of all facilities and addresses:
cdcr_address_path = directory_path.joinpath("data/cdcr.txt")

# CDCR PDF file path of all facilities and month-to-month population data:
cdcr_directory_path = directory_path.joinpath("data/CDCR_MONTHLY_POP_2020_2025")
cdcr_pdf_files = cdcr_directory_path.glob('cdcr_pop_20*/Tpop1d*.pdf')

# CDCR XLSX file path of all conservation camps and their addresses:
cdcr_cons_camp_path = directory_path.joinpath("data/cal_conservation_camps_address.xlsx")

# XLSX file path of all California ICE detention facilities and their addresses:
cal_ice_detention_path = directory_path.joinpath("data/cal_ice_detention_address.xlsx")

# TRAC Immigration XLSX File of Immigration Data including majority of month-to-month daily population data:
trac_immigration_path = directory_path.joinpath("data/TRAC_IMMIGRATION_RECORDS_2020_2025.xlsx")

## California Department of Corrections and Rehabilitation Data

### Converting CDCR PDF files using pdfplumber

In [4]:
# Each file's page 1 and 2 is converted into a dataframe and stored in page_one_df_list and page_two_df_list

page_one_df_list = []
page_two_df_list = []

for pdf_file in cdcr_pdf_files: 
    try:
        with pdfplumber.open(pdf_file) as pdf:
            if len(pdf.pages) >= 1: 
                pdf_table = pdf.pages[0].extract_tables()
                pdf_df = pd.DataFrame(pdf_table)
                pdf_df["File Name"] = pdf_file.name
                page_one_df_list.append(pdf_df)
            if len(pdf.pages) >= 2: 
                pdf_table = pdf.pages[1].extract_tables()
                pdf_df = pd.DataFrame(pdf_table)
                pdf_df["File Name"] = pdf_file.name
                page_two_df_list.append(pdf_df)
    except Error as e: 
        print(f'An error occured while processing pdf: {pdf_file} : {e}.')
        
print('Completed conversion process and pdf files for page 1 and 2 are stored in page_one_df_list and page_two_df_list.')

Completed conversion process and pdf files for page 1 and 2 are stored in page_one_df_list and page_two_df_list.


### Extracting the Facility Name, Population, and Date from each DataFrame

In [5]:
# California City Correctional Facility is the only facility that is on page one since it was owned by
# CoreCivic, and was leased by the CDCR from Dec 2013 - March 2024. 
# It then became leased by ICE  in 2025, where it now falls under as an ICE detention center as of 2025. 

month_year_pattern = r'Tpop1d(\d{2})(\d{2})\-*\d*\.pdf'
cal_city_facility_pattern = r'(California City Correctional Facility)\s*(\d\,*\d*)'

processed_df = []
for df in page_one_df_list: 
    df_temp = pd.DataFrame()
    date = None
    
    try: 
        file_name = df['File Name'].iloc[0]
        regex_list = re.findall(month_year_pattern, file_name)

        if regex_list: 
            year_str, month_str = regex_list[0]
            date = datetime.strptime(f'{month_str} {year_str}', '%m %y').strftime('%Y-%m')
        else:
            print(f"Warning: Could not extract date from filename: {file_name}.")
            continue
        
        if date:
            column3_list = df.iloc[0, 3]
            column3_string = column3_list[0]
            result = re.findall(cal_city_facility_pattern, column3_string)
            
            if result:
                df_temp = pd.DataFrame(result, columns=['Institution Name', 'Population'])
                df_temp['Population'] = df_temp['Population'].astype(str).str.replace(',', '', regex=True).astype(int)
                df_temp['Date'] = date
                processed_df.append(df_temp)   
                
    except Exception as e:
        print(f'Error occurred while processing data from dataframe {df["File Name"].iloc[0]}: {e}.')
        continue
        
if processed_df:
    assert len(processed_df) == 45
    cal_city_fac_data_frames = pd.concat(processed_df, ignore_index=True)
    
cal_city_fac_data_frames['Institution Name'] = cal_city_fac_data_frames['Institution Name'].replace('California City Correctional Facility',
                                                                                                   'California City Correctional Facility (CAC)')
cal_city_fac_data_frames.head(5)

,Institution Name,Population,Date
0,California City Correctional Facility (CAC),1009,2023-05
1,California City Correctional Facility (CAC),133,2023-09
2,California City Correctional Facility (CAC),197,2023-08
3,California City Correctional Facility (CAC),1367,2023-04
4,California City Correctional Facility (CAC),623,2023-06


In [6]:
# Depending on the year data was gathered, data was split by the CDCR into 'Male Institutions' and 'Female Institutions',
# resulting in this split data falling into Column 3 and Column 4, as opposed to all data falling in Column 3 
# after using pdfplumber.

date_pattern = r'As of Midnight (\w+) \d+, (\d+)'
facility_population_pattern = r'(.+?)\s+(\d[\d,\.]*)\s+.*'

processed_data_frames = []
for df in page_two_df_list:
    df_temporary = pd.DataFrame()
    try:
        date_list = df.iloc[0, 0]
        date_string = date_list[0]
        
        regex_list = re.findall(date_pattern, date_string)
        regex_values = regex_list[0]
        month_str, year_str = regex_values[0], regex_values[1]
        result_date = datetime.strptime(f'{month_str} {year_str}', '%B %Y').strftime('%Y-%m')
    except Exception as e: 
        print(f'An error occurred trying to obtain date from {df["File Name"].iloc[0]}: {e}.')
    
    try: 
        column3_list = df.iloc[0,3]
        column3_string = column3_list[0]
        
        column4_list = df.iloc[0, 4]
        column4_string = column4_list[0]
        
        if re.search('^Avenal State Prison', column3_string):
            result = re.findall(facility_population_pattern, column3_string)
            df_temporary = pd.DataFrame(result, columns=['Institution Name', 'Population'])
        
        elif re.search('^Male Institutions', column3_string):
            combined_text = column3_string + '\n' + column4_string
            result = re.findall(facility_population_pattern, combined_text)
            df_temporary = pd.DataFrame(result, columns=['Institution Name', 'Population'])
    
    except Exception as e:
        print(f'Error occurred while trying to process data from dataframe:{df["File Name"].iloc[0]}: {e}.')

    if not df_temporary.empty:
        df_temporary['Population'] = df_temporary['Population'].str.replace(',','', regex=True).astype(int)
        df_temporary['Date'] = result_date
        processed_data_frames.append(df_temporary)
    else: 
        print(f'No data was able to be extracted from dataframe: {df["File Name"].iloc[0]} ')

page_two_fac_df = pd.concat(processed_data_frames, ignore_index=True)

cdcr_init_df = pd.concat([cal_city_fac_data_frames, page_two_fac_df], ignore_index=True)
cdcr_init_df

,Institution Name,Population,Date
0,California City Correctional Facility (CAC),1009,2023-05
1,California City Correctional Facility (CAC),133,2023-09
2,California City Correctional Facility (CAC),197,2023-08
3,California City Correctional Facility (CAC),1367,2023-04
4,California City Correctional Facility (CAC),623,2023-06
...,...,...,...
2432,Male Total,88699,2021-03
2433,Central California Women's Facility (CCWF),2169,2021-03
2434,California Institution for Women (CIW),990,2021-03
2435,Folsom State Prison (FOL),45,2021-03


In [7]:
# Cleaning up dataframe so it removes 'Male Total' and 'Female Total' with 'Institution Total', and 
# ensures addition of Folsom Women's Facility (FWF) to the dataframe as this was a facility that was previously closed
# and does not exist in the current file.

cdcr_copy_df = cdcr_init_df.copy()
cdcr_split_date = cdcr_copy_df[cdcr_copy_df["Institution Name"].isin(["Male Total", "Female Total"])]["Date"].unique()

for date in cdcr_split_date: 
    male_total_df = cdcr_copy_df[(cdcr_copy_df["Date"] == date) & (cdcr_copy_df["Institution Name"] == "Male Total")]
    male_total = int(male_total_df.iloc[0]["Population"])
    
    female_total_df = cdcr_copy_df[(cdcr_copy_df["Date"] == date) & (cdcr_copy_df["Institution Name"] == "Female Total")]
    female_total = int(female_total_df.iloc[0]["Population"])

    institution_total_row = pd.DataFrame([{
        "Institution Name": "Institution Total",
        "Population": male_total + female_total,
        "Date": date
    }])

    cdcr_copy_df = pd.concat([cdcr_copy_df, institution_total_row], ignore_index=True)
    
    folsom_facilities = cdcr_copy_df[(cdcr_copy_df["Date"] == date) & (cdcr_copy_df["Institution Name"] == "Folsom State Prison (FOL)")]
    
    if len(folsom_facilities) == 2: 
        folsom_fem_fac = int(folsom_facilities["Population"].idxmin())

        cdcr_copy_df.loc[folsom_fem_fac, "Institution Name"] = "Folsom Women's Facility (FWF)"
        cdcr_copy_df = cdcr_copy_df.drop(
            cdcr_copy_df[
                (cdcr_copy_df["Date"] == date) &
                (cdcr_copy_df['Institution Name'].isin(["Male Total", "Female Total"]))
                ].index
        )
        
cdcr_copy_df

,Institution Name,Population,Date
0,California City Correctional Facility (CAC),1009,2023-05
1,California City Correctional Facility (CAC),133,2023-09
2,California City Correctional Facility (CAC),197,2023-08
3,California City Correctional Facility (CAC),1367,2023-04
4,California City Correctional Facility (CAC),623,2023-06
...,...,...,...
2412,Institution Total,92685,2021-04
2413,Institution Total,94103,2021-05
2414,Institution Total,90995,2021-01
2415,Institution Total,91279,2021-02


In [8]:
# Creating a Percent of Total Population column by manually adding monthly populations to verify values.
# Percent of Total Population (for that Month) = Facility Population (for that Month) / Institution Total (for that Month)

cdcr_fac_df = cdcr_copy_df[cdcr_copy_df['Institution Name'] != 'Institution Total'].copy()
monthly_fac_total = cdcr_fac_df.groupby('Date')['Population'].transform('sum')

cdcr_fac_df['Percent of Total Population'] = (cdcr_fac_df['Population']/monthly_fac_total) * 100
cdcr_fac_df

,Institution Name,Population,Date,Percent of Total Population
0,California City Correctional Facility (CAC),1009,2023-05,1.062239
1,California City Correctional Facility (CAC),133,2023-09,0.141654
2,California City Correctional Facility (CAC),197,2023-08,0.209296
3,California City Correctional Facility (CAC),1367,2023-04,1.444803
4,California City Correctional Facility (CAC),623,2023-06,0.657839
...,...,...,...,...
2388,Valley State Prison (VSP),2761,2021-03,2.945287
2389,Wasco State Prison (WSP),3026,2021-03,3.227974
2391,Central California Women's Facility (CCWF),2169,2021-03,2.313773
2392,California Institution for Women (CIW),990,2021-03,1.056079


### Reading CDCR Facilities Address File and Setting Consistent Values for Both DataFrames

In [9]:
#Creating the facilities and address text file into a dataframe

facility_names_address_pattern = r'(.+\(.+\))\,*\s*(.*\s*\s*.*)'
column_names = ['Institution Name', 'Address']

with open(cdcr_address_path, "r") as txtfile: 
    txtfile_content = txtfile.read()
    
regex_list = re.findall(facility_names_address_pattern, txtfile_content)


cdcr_address_df = pd.DataFrame(regex_list, columns=column_names)
cdcr_address_df["Address"] = cdcr_address_df["Address"].str.replace('\n', ' ')

In [10]:
# Replacing a list of 'Institution Names' with the same 'Institution Names' in cdcr_df
cdcr_address_df['Institution Name'] = cdcr_address_df['Institution Name'].replace(['Richard J. Donovan Correctional Facility (RJD)',
                                                               'Substance Abuse Treatment Facility and State Prison, Corcoran (SATF-CSP, Corcoran)',
                                                                                   'California Health Care Facility (CHCF)', 'California State Prison, Centinela (CEN)',
                                                                                  'Folsom State Prison (FSP)',
                                                                                  'Folsom Women\'s Facility (FWF)'],
                                                              ['RJ Donovan Correctional Facility (RJD)',
                                                               'California Substance Abuse Treatment Facility (SATF)', 
                                                              'California Health Care Facility - Stockton (CHCF)', 'Centinela State Prison (CEN)',
                                                              'Folsom State Prison (FOL)',
                                                              'Folsom Women’s Facility (FWF)'])
# Removing delimiters from cdcr_df and replacing 'Institution Names' for consistency
cdcr_fac_df['Institution Name'] = cdcr_fac_df['Institution Name'].replace(['California Men\'s Colony (CMC)', 'Central California Women\'s Facility (CCWF)', 'San Quentin State Prison (SQ)',
                                                                  'Folsom Women\'s Facility (FWF)'], 
                                                                  ['California Men’s Colony (CMC)', 'Central California Women’s Facility (CCWF)', 'San Quentin Rehabilitation Center (SQ)',
                                                                  'Folsom Women’s Facility (FWF)'])

# Adding closed facilities' addresses to cdcr_address_df
folsom_womens_fac_row = ['Folsom Women’s Facility (FWF)',"300 Prison Rd, Represa, CA 95671"]
chuckawalla_fac_row = ["Chuckawalla Valley State Prison (CVSP)", "19025 Wiley's Well Road, Blythe, CA 92225"]
ccc_fac_row = ["California Correctional Center (CCC)", "711-045 Center Road, Susanville, CA 96130"]
deuel_fac_row = ["Deuel Vocational Institution (DVI)", "23500 Kasson Rd, Tracy, CA 95304"]
cal_city_fac_row = ["California City Correctional Facility (CAC)", "22844 Virginia Blvd, California City, CA 93505"]

cdcr_address_df.loc[len(cdcr_address_df)] = folsom_womens_fac_row
cdcr_address_df.loc[len(cdcr_address_df)] = chuckawalla_fac_row 
cdcr_address_df.loc[len(cdcr_address_df)] = ccc_fac_row 
cdcr_address_df.loc[len(cdcr_address_df)] = deuel_fac_row
cdcr_address_df.loc[len(cdcr_address_df)] = cal_city_fac_row

# Replacing some addresses for better readability
cdcr_address_df['Address'] = cdcr_address_df['Address'].replace(['1 Kings Way Avenal, CA 93204', 
                                                                 'Highway 101 North * Soledad, CA 93960',
                                                                'San Quentin, CA 94964 (415) 454-1460	San Quentin Rehabilitation Center'], 
                                                                ['1 Kings Way, Avenal, CA 93204',
                                                                'US-101, Soledad, CA 93960',
                                                                '100 Main St, San Quentin, CA 94964'])

cdcr_address_df

,Institution Name,Address
0,Avenal State Prison (ASP),"#1 Kings Way Avenal, CA 93204"
1,California Correctional Institution (CCI),"24900 Highway 202 Tehachapi, CA 93561"
2,California Health Care Facility - Stockton (CHCF),"7707 Austin Road Stockton, CA 95215"
3,California Institution for Men (CIM),"14901 Central Avenue Chino, CA 91710"
4,California Institution for Women (CIW),"16756 Chino-Corona Road Corona, CA 92880"
5,California Men’s Colony (CMC),"Highway 1 San Luis Obispo, CA 93409"
6,California Medical Facility (CMF),"1600 California Dr. Vacaville, CA 95696"
7,California Rehabilitation Center (CRC),"5th Street & Western Norco, CA 92860"
8,"California State Prison, Corcoran (COR)","4001 King Avenue Corcoran, CA 93212"
9,"California State Prison, Los Angeles County (LAC)","44750 60th Street West Lancaster, CA 93536-7620"


### Geocoding Facilities' addresses and Merging data frames

In [11]:
# Geocoding addresses in cdcr_fac_address_df
geolocator = GoogleV3(api_key=GOOGLE_GEOCODE_API_KEY, timeout=20)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def geocode_location(df, geocoder): 
    '''
    Takes in a dataframe's address column and geocodes column using Nominatim geocode.
    
    Args: 
        df (pd.DataFrame object) : the dataframe with an Address column.
        geocoder (Nominatim and RateLimiter Object) : the instantiated Nominatim and RateLimiter Object geocoder.
        
    Returns: 
        csv : csv file that has added a location, latitude and longitude column of the dataframe's Address column.
    '''
    df['Location'] = df['Address'].apply(geocoder)
    df['Latitude'] = df['Location'].apply(lambda loc: loc.latitude if loc else None)
    df['Longitude'] = df['Location'].apply(lambda loc: loc.longitude if loc else None)
    
    return df.to_csv(f'{directory_path}/data/geocoded_ice_det_results.csv', index=False) 

cdcr_geocoded_df = pd.read_csv(f'{directory_path}/data/geocoded_cdcr_results.csv')
cdcr_geocoded_df.head(10)

,Institution Name,Address,Location,Latitude,Longitude
0,Avenal State Prison (ASP),"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364
1,California Correctional Institution (CCI),"24900 Highway 202 Tehachapi, CA 93561","24900 Valley Boulevard, Kern County, Californi...",35.108417,-118.572037
2,California Health Care Facility - Stockton (CHCF),"7707 Austin Road Stockton, CA 95215","Austin Road, San Joaquin County, California, 9...",37.897177,-121.186537
3,California Institution for Men (CIM),"14901 Central Avenue Chino, CA 91710","14901, Central Avenue, Chino, San Bernardino C...",33.982548,-117.688761
4,California Institution for Women (CIW),"16756 Chino-Corona Road Corona, CA 92880","California Institution for Women, 16756, Chino...",33.949354,-117.636656
5,California Men’s Colony (CMC),"Highway 1 San Luis Obispo, CA 93409","1, Santa Rosa Street, Railroad District, San L...",35.277985,-120.655209
6,California Medical Facility (CMF),"1600 California Dr. Vacaville, CA 95696","California Medical Facility, 1600, California ...",38.328111,-121.978759
7,California Rehabilitation Center (CRC),"5th Street & Western Norco, CA 92860","5th Street, Riverside County, California, 9286...",33.928706,-117.572279
8,"California State Prison, Corcoran (COR)","4001 King Avenue Corcoran, CA 93212","4001, King Avenue, Corcoran, Kings County, Cal...",36.058046,-119.554325
9,"California State Prison, Los Angeles County (LAC)","44750 60th Street West Lancaster, CA 93536-7620","California State Prison, Los Angeles County (L...",34.693346,-118.228780


In [12]:
# Manually inputting geocoded information of addresses that could not be read/found by the geocoder via Nominatim.

missing_indices = [0, 1, 2, 7, 20, 25, 30, 34]
columns_to_update = ['Location', 'Latitude', 'Longitude']

corrected_missing_rows = [["Kings Way, Hanford, California, 93245, United States of America", 35.975052, -120.116364],
                          ["24900 Valley Boulevard, Kern County, California, 93561, United States of America", 35.1084168, -118.5720369],
                          ["Austin Road, San Joaquin County, California, 95215, United States of America", 37.8971772, -121.186537],
                          ["5th Street, Riverside County, California, 92860, United States of America", 33.9287061, -117.5722791],
                          ["4001 Twin Cities Road, Amador County, California, 95640, United States of America", 38.3734506, -120.9513007],
                          ["31625, Soledad, Monterey County, California, 93960, United States of America", 36.4786606, -121.3743167],
                          ["701 Perimeter Road, Kern County, California, 93280, United States of America", 35.5918171, -119.4078166],
                          ["Kasson Road, San Joaquin County, California, 95304, United States of America", 37.7479842, -121.3293874]]

cdcr_geocoded_df.loc[missing_indices, columns_to_update] = corrected_missing_rows

In [50]:
# Merging cdcr dataframe with facility population and cdcr dataframe with geocoded addresses
cdcr_fac_address_df = pd.merge(cdcr_fac_df, cdcr_geocoded_df, on='Institution Name', how='outer')

cdcr_fac_address_df['Facility Type'] = 'Adult Facility'
cdcr_fac_address_df.head(5)

,Institution Name,Population,Date,Percent of Total Population,Address,Location,Latitude,Longitude,Facility Type
0,Avenal State Prison (ASP),4230,2024-09,4.657616,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility
1,Avenal State Prison (ASP),4431,2024-08,4.882322,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility
2,Avenal State Prison (ASP),4056,2024-10,4.473858,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility
3,Avenal State Prison (ASP),4717,2024-01,5.103266,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility
4,Avenal State Prison (ASP),4796,2024-03,5.214631,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility


## CDCR Conservation Camps Data

In [14]:
# Geocoding CDCR Conservation Camp from Address' Column via Google API Key
# Reading from saved geocoded locations file (from an earlier run)
cdcr_cons_camp_df = pd.read_csv(f'{directory_path}/data/geocoded_cons_camp_results.csv')

# Adding date range for Conservation Camp facilities
date_range = pd.date_range(start='2020-01', end='2025-09', freq='MS') 
dates_df = pd.DataFrame({'Date': date_range})
cdcr_cons_camp_df = cdcr_cons_camp_df.merge(dates_df, how='cross') 
cdcr_cons_camp_df['Date'] = cdcr_cons_camp_df['Date'].dt.strftime('%Y-%m')

cdcr_cons_camp_df.insert(1, 'Population', -1.0)
cdcr_cons_camp_df.insert(3, 'Percent of Total Population', -1.0)
cdcr_cons_camp_df['Facility Type'] = 'Conservation Camp'
cdcr_cons_camp_df

,Institution Name,Population,Address,Percent of Total Population,Location,Latitude,Longitude,Date,Facility Type
0,Acton #11,-1.0,"8800 Soledad Canyon Road, Acton, CA 93510",-1.0,"8800 Soledad Canyon Rd, Acton, CA 93510, USA",34.437008,-118.287560,2020-01,Conservation Camp
1,Acton #11,-1.0,"8800 Soledad Canyon Road, Acton, CA 93510",-1.0,"8800 Soledad Canyon Rd, Acton, CA 93510, USA",34.437008,-118.287560,2020-02,Conservation Camp
2,Acton #11,-1.0,"8800 Soledad Canyon Road, Acton, CA 93510",-1.0,"8800 Soledad Canyon Rd, Acton, CA 93510, USA",34.437008,-118.287560,2020-03,Conservation Camp
3,Acton #11,-1.0,"8800 Soledad Canyon Road, Acton, CA 93510",-1.0,"8800 Soledad Canyon Rd, Acton, CA 93510, USA",34.437008,-118.287560,2020-04,Conservation Camp
4,Acton #11,-1.0,"8800 Soledad Canyon Road, Acton, CA 93510",-1.0,"8800 Soledad Canyon Rd, Acton, CA 93510, USA",34.437008,-118.287560,2020-05,Conservation Camp
...,...,...,...,...,...,...,...,...,...
2479,Washington Ridge #44,-1.0,"11425 Conservation Camp Road, Nevada City, CA ...",-1.0,"11425 Conservation Rd, Nevada City, CA 95959, USA",39.308022,-120.934263,2025-05,Conservation Camp
2480,Washington Ridge #44,-1.0,"11425 Conservation Camp Road, Nevada City, CA ...",-1.0,"11425 Conservation Rd, Nevada City, CA 95959, USA",39.308022,-120.934263,2025-06,Conservation Camp
2481,Washington Ridge #44,-1.0,"11425 Conservation Camp Road, Nevada City, CA ...",-1.0,"11425 Conservation Rd, Nevada City, CA 95959, USA",39.308022,-120.934263,2025-07,Conservation Camp
2482,Washington Ridge #44,-1.0,"11425 Conservation Camp Road, Nevada City, CA ...",-1.0,"11425 Conservation Rd, Nevada City, CA 95959, USA",39.308022,-120.934263,2025-08,Conservation Camp


## California ICE Detention Facilities CSV Data

In [65]:
# Geocoding ICE Detention Facilities from Address' column via Google API Key and merging with TRAC Immigration data file
# Reading from saved geocoded locations file (from an earlier run)
cal_ice_detention_df = pd.read_csv(f'{directory_path}/data/geocoded_ice_det_results.csv')

# Cleaning TRAC Immigration dataset to be able to merge with cal_ice_detention_df
trac_df = pd.read_excel(f'{directory_path}/data/TRAC_IMMIGRATION_RECORDS_2020_2025.xlsx')
trac_df['Name Clean'] = trac_df['Name'].str.upper().str.strip()

facility_name_map = {
    'ADELANTO ICE PROCESSING CENTER': 'Adelanto ICE Processing Center',
    
    'DESERT VIEW ANNEX': 'Desert View Annex',
    'DESERT VIEW': 'Desert View Annex',
    
    'MESA VERDE ICE PROCESSING CENTER': 'Mesa Verde ICE Processing Facility',
    'MESA VERDE ICE PROCESSING FACILITY': 'Mesa Verde ICE Processing Facility',
    
    'GOLDEN STATE ANNEX': 'Golden State Annex',
    
    'IMPERIAL REGIONAL DETENTION FACILITY': 'Imperial Regional Detention Facility',
    
    'OTAY MESA DETENTION CENTER': 'Otay Mesa Detention Center',
    'OTAY MESA DETENTION CENTER (SAN DIEGO CDF)': 'Otay Mesa Detention Center',
    
    'CALIFORNIA CITY CORRECTIONAL CENTER': 'California City Correctional Facility'
}

trac_df['Institution Name'] = trac_df['Name Clean'].map(facility_name_map)
trac_df = trac_df.dropna(subset=['Institution Name'])
trac_df_clean = trac_df.iloc[:, [6, 7, 9]]
trac_df_clean = trac_df_clean.rename(columns={
    'Average Daily Population': 'Population',
    'Current as of**': 'Date'
})

# Merging with geocoded addresses of set ICE Detention Facilities
ice_detention_df = pd.merge(cal_ice_detention_df, trac_df_clean, how='left', on='Institution Name')
ice_detention_df['Date'] = pd.to_datetime(ice_detention_df['Date'])
ice_detention_df['YearMonth'] = ice_detention_df['Date'].dt.to_period('M')

# Cleaning for multiple population data in a given month and missing data from January - July, 2020 and October, 2020
ice_monthly = ice_detention_df.groupby(['Institution Name', 'YearMonth'])['Population'].mean().reset_index()
ice_monthly['Population'] = np.floor(ice_monthly['Population'])

monthly_date_range = pd.period_range(start='2020-01', end='2025-01', freq='M')
ice_fac_names = ice_monthly['Institution Name'].unique()
idx = pd.MultiIndex.from_product([ice_fac_names, monthly_date_range], names=['Institution Name', 'YearMonth'])

# Merging with geocoded addresses as they were dropped when aggregating by Institution Name and YearMonth
ice_clean_df = ice_monthly.set_index(['Institution Name', 'YearMonth']).reindex(idx).reset_index()
ice_clean_df['Population'] = ice_clean_df['Population'].fillna(-1)
ice_clean_df = pd.merge(ice_clean_df, cal_ice_detention_df, on='Institution Name', how='left')

# Calculating Percent of Total Population in reference to ICE Detention Facilities
known_population_df = ice_clean_df[ice_clean_df['Population'] > 0]
known_population_index = known_population_df.index
pop_monthly_tot = known_population_df.groupby('YearMonth')['Population'].transform('sum')

ice_clean_df['Percent of Total Population'] = -1.0
ice_clean_df.iloc[known_population_index, 7] = (ice_clean_df.iloc[known_population_index, 2] / pop_monthly_tot) * 100
ice_clean_df['Facility Type'] = 'ICE Detention'
ice_clean_df = ice_clean_df.rename(columns={
    'YearMonth': 'Date'
})
ice_clean_df.head(5)

,Institution Name,Date,Population,Address,Location,Latitude,Longitude,Percent of Total Population,Facility Type
0,Adelanto ICE Processing Center,2020-01,-1.0,"Adelanto ICE Processing Center - West, 10250 R...","10250 Rancho Rd, Adelanto, CA 92301, USA",34.559101,-117.441495,-1.0,ICE Detention
1,Adelanto ICE Processing Center,2020-02,-1.0,"Adelanto ICE Processing Center - West, 10250 R...","10250 Rancho Rd, Adelanto, CA 92301, USA",34.559101,-117.441495,-1.0,ICE Detention
2,Adelanto ICE Processing Center,2020-03,-1.0,"Adelanto ICE Processing Center - West, 10250 R...","10250 Rancho Rd, Adelanto, CA 92301, USA",34.559101,-117.441495,-1.0,ICE Detention
3,Adelanto ICE Processing Center,2020-04,-1.0,"Adelanto ICE Processing Center - West, 10250 R...","10250 Rancho Rd, Adelanto, CA 92301, USA",34.559101,-117.441495,-1.0,ICE Detention
4,Adelanto ICE Processing Center,2020-05,-1.0,"Adelanto ICE Processing Center - West, 10250 R...","10250 Rancho Rd, Adelanto, CA 92301, USA",34.559101,-117.441495,-1.0,ICE Detention


In [66]:
# Merging all dataframes into one facilities dataframe
all_facilities_df = pd.concat([cdcr_fac_address_df, cdcr_cons_camp_df, ice_clean_df])
all_facilities_df.head(5)

,Institution Name,Population,Date,Percent of Total Population,Address,Location,Latitude,Longitude,Facility Type
0,Avenal State Prison (ASP),4230.0,2024-09,4.657616,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility
1,Avenal State Prison (ASP),4431.0,2024-08,4.882322,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility
2,Avenal State Prison (ASP),4056.0,2024-10,4.473858,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility
3,Avenal State Prison (ASP),4717.0,2024-01,5.103266,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility
4,Avenal State Prison (ASP),4796.0,2024-03,5.214631,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.975052,-120.116364,Adult Facility


## Merging Each Facility's Polygon Data with the Consolidated Facilities DataFrame

In [77]:
# Reading facilities geopackage created via QGIS. 
facilities_polygons = gpd.read_file(ca_facilities_geo_path)

# Calculating centroid from the facilities' geometry column
facilities_meters = facilities_polygons.to_crs(epsg=3310)
facilities_meters['centroid_geom'] = facilities_meters.geometry.centroid
facilities_polygons['Centroid'] = facilities_meters['centroid_geom'].to_crs(epsg=4326)

# Merging with all_facilities_df
final_facilities_data = pd.merge(all_facilities_df, facilities_polygons, how='left', on='Institution Name')

# Converting to a GeoDataFrame
final_facilities_data = gpd.GeoDataFrame(
    final_facilities_data, 
    geometry='geometry', 
    crs="EPSG:4326"
)

# Renaming and setting updated latitude and longitude values.
final_facilities_data = final_facilities_data.rename(columns={
    'id': 'ID',
    'area': 'Area',
    'perimeter': 'Perimeter',
    'geometry': 'Geometry'
})
final_facilities_data['Latitude'] = final_facilities_data['Centroid'].y
final_facilities_data['Longitude'] = final_facilities_data['Centroid'].x

final_facilities_data
final_facilities_data.to_pickle("final_facilities_data.pkl")

,Institution Name,Population,Date,Percent of Total Population,Address,Location,Latitude,Longitude,Facility Type,ID,Area,Perimeter,Geometry,Centroid
0,Avenal State Prison (ASP),4230.0,2024-09,4.657616,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.973954,-120.123069,Adult Facility,37,409659.422287,2361.498893,"POLYGON ((-120.12207 35.97033, -120.12274 35.9...",POINT (-120.12307 35.97395)
1,Avenal State Prison (ASP),4431.0,2024-08,4.882322,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.973954,-120.123069,Adult Facility,37,409659.422287,2361.498893,"POLYGON ((-120.12207 35.97033, -120.12274 35.9...",POINT (-120.12307 35.97395)
2,Avenal State Prison (ASP),4056.0,2024-10,4.473858,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.973954,-120.123069,Adult Facility,37,409659.422287,2361.498893,"POLYGON ((-120.12207 35.97033, -120.12274 35.9...",POINT (-120.12307 35.97395)
3,Avenal State Prison (ASP),4717.0,2024-01,5.103266,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.973954,-120.123069,Adult Facility,37,409659.422287,2361.498893,"POLYGON ((-120.12207 35.97033, -120.12274 35.9...",POINT (-120.12307 35.97395)
4,Avenal State Prison (ASP),4796.0,2024-03,5.214631,"#1 Kings Way Avenal, CA 93204","Kings Way, Hanford, California, 93245, United ...",35.973954,-120.123069,Adult Facility,37,409659.422287,2361.498893,"POLYGON ((-120.12207 35.97033, -120.12274 35.9...",POINT (-120.12307 35.97395)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5252,Otay Mesa Detention Center,1241.0,2024-09,46.971991,"Otay Mesa Detention Center, 7488 Calzada De La...","7488 Calzada De La Fuente, San Diego, CA 92154...",32.575841,-116.915346,ICE Detention,73,26231.019040,657.070942,"POLYGON ((-116.9148 32.57526, -116.91645 32.57...",POINT (-116.91535 32.57584)
5253,Otay Mesa Detention Center,1320.0,2024-10,44.639838,"Otay Mesa Detention Center, 7488 Calzada De La...","7488 Calzada De La Fuente, San Diego, CA 92154...",32.575841,-116.915346,ICE Detention,73,26231.019040,657.070942,"POLYGON ((-116.9148 32.57526, -116.91645 32.57...",POINT (-116.91535 32.57584)
5254,Otay Mesa Detention Center,1364.0,2024-11,45.603477,"Otay Mesa Detention Center, 7488 Calzada De La...","7488 Calzada De La Fuente, San Diego, CA 92154...",32.575841,-116.915346,ICE Detention,73,26231.019040,657.070942,"POLYGON ((-116.9148 32.57526, -116.91645 32.57...",POINT (-116.91535 32.57584)
5255,Otay Mesa Detention Center,1373.0,2024-12,45.873705,"Otay Mesa Detention Center, 7488 Calzada De La...","7488 Calzada De La Fuente, San Diego, CA 92154...",32.575841,-116.915346,ICE Detention,73,26231.019040,657.070942,"POLYGON ((-116.9148 32.57526, -116.91645 32.57...",POINT (-116.91535 32.57584)
